In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Black vs SABR Swaption Analysis

This notebook compares flat-vol Black pricing with SABR-adjusted pricing for a USD European payer swaption. It also shows a simple SABR calibration workflow on a synthetic smile.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import pandas as pd
import matplotlib.pyplot as plt

from swaption_pricing.black76 import price_swaption
from swaption_pricing.calibration import calibrate_sabr_to_vols
from swaption_pricing.risk import compare_black_and_sabr_risk
from swaption_pricing.sabr import SabrParams, price_swaption_with_sabr, sabr_implied_volatility
from swaption_pricing.swap import forward_swap_rate
from swaption_pricing.types import CurvePoint, SwaptionSpec

In [ ]:
curve = [
    CurvePoint(maturity=1.0, zero_rate=0.0420),
    CurvePoint(maturity=2.0, zero_rate=0.0415),
    CurvePoint(maturity=3.0, zero_rate=0.0410),
    CurvePoint(maturity=4.0, zero_rate=0.0408),
    CurvePoint(maturity=5.0, zero_rate=0.0405),
    CurvePoint(maturity=6.0, zero_rate=0.0403),
    CurvePoint(maturity=7.0, zero_rate=0.0402),
]

expiry = 2.0
tenor = 5.0
pay_frequency = 1
notional = 10_000_000.0
flat_black_vol = 0.20
sabr_params = SabrParams(alpha=0.0200, beta=0.50, rho=-0.25, nu=0.40)
forward = forward_swap_rate(curve, expiry, tenor, pay_frequency)
strikes = [0.0300, 0.0350, 0.0400, 0.0450, 0.0500]
forward

In [ ]:
rows = []
for strike in strikes:
    spec = SwaptionSpec(
        notional=notional,
        expiry=expiry,
        tenor=tenor,
        strike=strike,
        pay_frequency=pay_frequency,
        option_type='payer',
    )
    black_price = price_swaption(curve, spec, flat_black_vol)
    sabr_price, sabr_vol = price_swaption_with_sabr(curve, spec, sabr_params)
    rows.append({
        'strike': strike,
        'forward': forward,
        'black_vol': flat_black_vol,
        'sabr_vol': sabr_vol,
        'black_price': black_price,
        'sabr_price': sabr_price,
    })

comparison = pd.DataFrame(rows)
comparison

In [ ]:
ax = comparison.plot(x='strike', y=['black_vol', 'sabr_vol'], marker='o', figsize=(8, 4), title='Volatility Comparison')
ax.set_ylabel('Implied Volatility')
plt.show()

ax = comparison.plot(x='strike', y=['black_price', 'sabr_price'], marker='o', figsize=(8, 4), title='Price Comparison')
ax.set_ylabel('Swaption Price')
plt.show()

In [ ]:
atm_spec = SwaptionSpec(
    notional=notional,
    expiry=expiry,
    tenor=tenor,
    strike=0.0400,
    pay_frequency=pay_frequency,
    option_type='payer',
)

risk_comparison = compare_black_and_sabr_risk(curve, atm_spec, black_vol=flat_black_vol, sabr_params=sabr_params)
pd.DataFrame([
    {
        'model': 'Black',
        'price': risk_comparison.black_price,
        'vol': risk_comparison.black_vol,
        'pv01': risk_comparison.black_risk.pv01,
        'vega_or_alpha_risk': risk_comparison.black_risk.vega,
        'theta': risk_comparison.black_risk.theta,
    },
    {
        'model': 'SABR-adjusted',
        'price': risk_comparison.sabr_price,
        'vol': risk_comparison.sabr_vol,
        'pv01': risk_comparison.sabr_risk.pv01,
        'vega_or_alpha_risk': risk_comparison.sabr_risk.vega,
        'theta': risk_comparison.sabr_risk.theta,
    },
])

In [ ]:
synthetic_market_vols = [sabr_implied_volatility(forward, strike, expiry, sabr_params) for strike in strikes]
calibration = calibrate_sabr_to_vols(
    forward=forward,
    strikes=strikes,
    expiry=expiry,
    market_vols=synthetic_market_vols,
    beta=0.50,
    initial_guess=(0.0180, -0.10, 0.30),
)

pd.DataFrame({
    'strike': calibration.strikes,
    'market_vol': calibration.market_vols,
    'fitted_vol': calibration.fitted_vols,
})

In [ ]:
calibration.params